# 04b · ¿Otras fuentes de datos mejorarían el modelo? Prueba final e importancia de variables

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** El modelo del notebook 04 usa 34 variables. Antes de cerrar queremos saber dos cosas:

1. **¿Nos faltó información?** Probamos 18 variables nuevas de otras fuentes: INEGI (censos completos y mapa de municipios) e INE (casillas). Para cada una escribimos antes la hipótesis de por qué podría ayudar.
2. **¿Qué pesa de verdad en el modelo?** Medimos la importancia de cada variable con **permutación**: cuánto empeora la predicción de una elección que el modelo no vio si "revolvemos" esa variable entre municipios.

**Regla para aceptar una variable nueva (fijada antes de ver resultados):** que vaya con lo que la historia no anticipa en las dos transiciones **y** que, al agregarla al modelo, el error baje en las dos validaciones (aprender de 2015 y predecir 2021, y al revés).

## Resumen de lo que encontramos

1. **12 de las 18 variables nuevas se relacionan de forma estable** con lo que la historia no anticipa. Las más fuertes son la densidad de población, la participación económica de las mujeres, tener radio y los electores por casilla.
2. **Pero casi ninguna mejora la predicción.** Solo dos mejoran en las dos validaciones, y muy poco:
   - la **densidad de población** baja el error 0.05 y 0.03 puntos (de 5.49 y 6.65);
   - el **promedio de ocupantes por vivienda**, 0.001 y 0.03.

   Las demás empeoran al menos una de las dos validaciones. **El modelo ya captura esa información** con las variables que tiene: población, rural, escolaridad y pobreza.
3. **Un modelo no lineal no la aprovecha mejor:** gradient boosting se equivoca mucho más que Huber (7.3 contra 5.5 puntos al predecir 2021), con o sin las variables nuevas.
4. **Lo que pesa de verdad**, en este orden:
   - la posición del municipio en la presidencial anterior;
   - si elige ayuntamiento el mismo día;
   - el conjunto de variables sociales y económicas;
   - la violencia y la competencia política casi no pesan.
5. **Recomendación (D20, pendiente):** no cambiar el modelo. La mejora de la densidad es real pero diminuta. Es preferible un modelo más simple y ya validado.

## Hipótesis, escritas antes de probar

| # | Fuente | Variable | Por qué podría ayudar |
|---|---|---|---|
| H1 | INEGI, censos 2010 y 2020 (ITER) | % sin religión, % católica | Las redes comunitarias, entre ellas las religiosas, movilizan el voto |
| H2 | ITER | Desocupación; participación económica, total y de mujeres | El empleo conecta con la vida pública; el desempleo aleja |
| H3 | ITER | % de hogares con jefatura femenina | Señal de migración masculina; el sentido es incierto |
| H4 | ITER | % de casados y de solteros (12 años o más) | Arraigo |
| H5 | ITER | % con discapacidad o limitación | Barreras para llegar a la casilla |
| H6 | ITER | % de viviendas con automóvil, radio, TV o teléfono fijo | Movilidad y acceso a información de campaña |
| H7 | ITER | % de analfabetismo | Barrera para informarse |
| H8 | ITER | % de viviendas deshabitadas | Abandono y migración: lista con gente que ya no vive ahí |
| H9 | ITER | Promedio de ocupantes por vivienda | Hacinamiento; el sentido es incierto |
| H10 | INE, actas | Electores por casilla | Casillas más llenas: filas y distancia |
| H11 | INEGI, mapa de municipios | Densidad de población | Aislamiento frente a la ciudad |

**Sin información del futuro:** para 2015 usamos el censo 2010 y las casillas de 2012; para 2021, el censo 2020 y las casillas de 2018; para 2027, el censo 2020 y las casillas de 2024. La densidad usa el área de los municipios de 2024 (los límites casi no cambian).

**Advertencia:** la pregunta de discapacidad cambió entre censos ("limitación" en 2010, "discapacidad" en 2020). Como cada transición usa su propio censo, la comparación es dentro de cada año.

In [1]:
from pathlib import Path
import csv
import hashlib
import io
import warnings
import zipfile

import geopandas as gpd
import numpy as np
import polars as pl
from scipy.stats import rankdata, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(200)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
P = raiz / 'data/processed'
ITER = {2010: ('data/raw/externos/inegi/2010/iter_nal_2010_csv.zip', 'iter_00_cpv2010/conjunto_de_datos/iter_00_cpv2010.csv'),
        2020: ('data/raw/externos/inegi/2020/iter_00_cpv2020_csv.zip', 'iter_00_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv')}
with (raiz / 'data/reference/huellas_fuentes_externas.csv').open(encoding='utf-8') as f:
    huellas = {r['archivo_relativo']: r['sha256'] for r in csv.DictReader(f)}
for z, _ in ITER.values():
    with (raiz / z).open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == huellas[z], f'{z} no coincide con su huella'
assert (P / 'variables_transiciones.parquet').exists(), 'Primero corre el notebook 03b'
variables = pl.read_parquet(P / 'variables_transiciones.parquet')
rng = np.random.default_rng(2027)
print('Fuentes verificadas contra sus huellas.')

Fuentes verificadas contra sus huellas.


## 1 · Construimos las variables nuevas

In [2]:
def iter_municipal(anio):
    z, miembro = ITER[anio]
    d = pl.read_csv(io.BytesIO(zipfile.ZipFile(raiz / z).read(miembro)), infer_schema=False, encoding='utf8-lossy')
    d = d.rename({c: c.lower().replace('﻿', '') for c in d.columns}).filter((pl.col('loc') == '0000') & (pl.col('mun') != '000'))
    n = lambda c: pl.col(c).cast(pl.Float64, strict=False)
    return d.select(
        cvegeo=pl.col('entidad') + pl.col('mun'),
        pct_sin_religion=n('psin_relig') / n('pobtot') * 100,
        pct_catolica=n('pcatolica') / n('pobtot') * 100,
        pct_desocupacion=n('pdesocup') / n('pea') * 100,
        tasa_participacion_economica=n('pea') / n('p_12ymas') * 100,
        tasa_participacion_economica_mujeres=n('pea_f') / n('p_12ymas_f') * 100,
        pct_hogares_jefatura_femenina=n('hogjef_f') / n('tothog') * 100,
        pct_casados_12mas=n('p12ym_casa') / n('p_12ymas') * 100,
        pct_solteros_12mas=n('p12ym_solt') / n('p_12ymas') * 100,
        pct_discapacidad_limitacion=n('pcon_lim' if anio == 2010 else 'pcon_disc') / n('pobtot') * 100,
        pct_viviendas_automovil=n('vph_autom') / n('tvivparhab') * 100,
        pct_viviendas_radio=n('vph_radio') / n('tvivparhab') * 100,
        pct_viviendas_tv=n('vph_tv') / n('tvivparhab') * 100,
        pct_viviendas_telefono_fijo=n('vph_telef') / n('tvivparhab') * 100,
        pct_analfabetismo=n('p15ym_an') / n('p_15ymas') * 100,
        pct_viviendas_deshabitadas=n('vivpar_des') / n('tvivpar') * 100,
        promedio_ocupantes_vivienda=n('prom_ocup'),
        poblacion_censo=n('pobtot'))
censos = {a: iter_municipal(a) for a in ITER}

secciones = pl.read_parquet(P / 'secciones_territorio_2024.parquet')
def electores_por_casilla(anio):
    lista = (pl.scan_parquet(P / f'ccpc/ccpc-{anio}.parquet').group_by(pl.col('EDOCVE').cast(pl.Int64), 'SECCION').agg(pl.col('LN').sum()).collect())
    s = secciones.filter(pl.col('AELEC') == anio, pl.col('hay_actas').fill_null(False)).with_columns(pl.col('EDOCVE').cast(pl.Int64)).join(lista, on=['EDOCVE', 'SECCION'])
    return s.group_by(pl.col('municipio_2024').alias('cvegeo')).agg(electores_por_casilla=pl.col('LN').sum() / pl.col('actas_casillas').sum())
casillas = {a: electores_por_casilla(a) for a in (2012, 2018, 2024)}

mapa = gpd.read_file(P / 'municipios_inegi.geojson')
albers = '+proj=aea +lat_1=14.5 +lat_2=32.5 +lat_0=24 +lon_0=-105 +datum=WGS84 +units=m'   # proyección de área igual para México
area = pl.DataFrame({'cvegeo': mapa['CVEGEO'].astype(str).tolist(), 'km2': (mapa.to_crs(albers).geometry.area / 1e6).tolist()})

CENSO, ACTAS = {2015: 2010, 2021: 2020, 2027: 2020}, {2015: 2012, 2021: 2018, 2027: 2024}
NUEVAS = [c for c in censos[2010].columns if c not in ('cvegeo', 'poblacion_censo')] + ['electores_por_casilla', 'densidad_log']
ampliada = pl.concat([variables.filter(pl.col('eleccion_objetivo') == T).join(censos[CENSO[T]], on='cvegeo', how='left')
                      .join(casillas[ACTAS[T]], on='cvegeo', how='left').join(area, on='cvegeo', how='left')
                      .with_columns(densidad_log=(pl.col('poblacion_censo') / pl.col('km2')).log()) for T in (2015, 2021, 2027)], how='diagonal_relaxed')
no_finitos = {c: ampliada.select((~pl.col(c).is_finite()).sum()).item() for c in NUEVAS}
ampliada = ampliada.with_columns([pl.when(pl.col(c).is_finite()).then(pl.col(c)).alias(c) for c in NUEVAS])
print('Valores no finitos que pasan a vacío:', {k: v for k, v in no_finitos.items() if v})
print(f'Área total de los municipios: {area["km2"].sum():,.0f} km² (México: unos 1.96 millones)')
ampliada.group_by('eleccion_objetivo').agg(pl.len().alias('Municipios'), *[pl.col(c).null_count().alias(f'vacíos: {c}') for c in ['pct_sin_religion', 'electores_por_casilla', 'densidad_log']]).sort('eleccion_objetivo')

Valores no finitos que pasan a vacío: {'electores_por_casilla': 3}
Área total de los municipios: 1,962,292 km² (México: unos 1.96 millones)


eleccion_objetivo,Municipios,vacíos: pct_sin_religion,vacíos: electores_por_casilla,vacíos: densidad_log
i16,u32,u32,u32,u32
2015,2477,21,4,21
2021,2477,8,0,8
2027,2477,8,2,8


Los municipios sin dato del censo son los que se crearon después (21 en el censo 2010, 8 en el de 2020). El modelo los completa con la mediana, igual que en el notebook 04.

## 2 · ¿Van con lo que la historia no anticipa?

Igual que en el notebook 03b (sección 5): correlación de Spearman con el residuo de la intermedia dado la presidencial anterior, con intervalo de confianza de 95 % por bootstrap. "Estable" = mismo signo y distinta de cero en las dos transiciones.

In [3]:
def con_resultado(T):
    return ampliada.filter((pl.col('eleccion_objetivo') == T) & pl.col('desviacion_objetivo').is_not_null()
                           & ~pl.col('sin_casilla_objetivo').fill_null(False) & pl.col('desviacion_presidencial').is_not_null())
def spearman_ic(x, y, veces=400):
    ok = ~(np.isnan(x) | np.isnan(y)); x, y = x[ok], y[ok]
    bs = [spearmanr(x[i], y[i]).statistic for i in (rng.integers(0, len(x), len(x)) for _ in range(veces))]
    return spearmanr(x, y).statistic, np.percentile(bs, 2.5), np.percentile(bs, 97.5)
filas = []
for T in (2015, 2021):
    d = con_resultado(T)
    pendiente, ordenada = np.polyfit(d['desviacion_presidencial'], d['desviacion_objetivo'], 1)
    residuo = d['desviacion_objetivo'].to_numpy() - (ordenada + pendiente * d['desviacion_presidencial'].to_numpy())
    for v in NUEVAS:
        r, bajo, alto = spearman_ic(d[v].cast(pl.Float64).to_numpy(), residuo)
        filas.append({'variable': v, 'T': T, 'txt': f'{r:.2f} [{bajo:.2f}, {alto:.2f}]', 'bajo': bajo, 'alto': alto, 'r': r})
estabilidad = (pl.DataFrame(filas).group_by('variable').agg(pl.col('txt').sort_by('T'), pl.col('r').sort_by('T'),
               ((pl.col('bajo') > 0).all() | (pl.col('alto') < 0).all()).alias('estable'))
               .with_columns(pl.col('txt').list.get(0).alias('2012 → 2015: ρ [IC 95 %]'), pl.col('txt').list.get(1).alias('2018 → 2021: ρ [IC 95 %]'),
                             pl.col('r').list.mean().abs().alias('fuerza'))
               .sort('estable', 'fuerza', descending=True))
estabilidad.select('variable', '2012 → 2015: ρ [IC 95 %]', '2018 → 2021: ρ [IC 95 %]', pl.col('estable').replace_strict({True: 'sí', False: 'no'}).alias('¿Estable en las dos?'))

variable,2012 → 2015: ρ [IC 95 %],2018 → 2021: ρ [IC 95 %],¿Estable en las dos?
str,str,str,str
"""densidad_log""","""-0.30 [-0.34, -0.27]""","""-0.19 [-0.22, -0.15]""","""sí"""
"""tasa_participacion_economica_mujeres""","""-0.19 [-0.23, -0.16]""","""-0.18 [-0.22, -0.14]""","""sí"""
"""pct_viviendas_radio""","""-0.21 [-0.25, -0.18]""","""-0.13 [-0.17, -0.09]""","""sí"""
"""tasa_participacion_economica""","""-0.20 [-0.24, -0.15]""","""-0.13 [-0.16, -0.09]""","""sí"""
"""electores_por_casilla""","""-0.18 [-0.22, -0.14]""","""-0.14 [-0.17, -0.10]""","""sí"""
"""pct_catolica""","""0.22 [0.19, 0.26]""","""0.08 [0.04, 0.12]""","""sí"""
"""pct_casados_12mas""","""0.08 [0.04, 0.12]""","""0.19 [0.15, 0.23]""","""sí"""
"""pct_analfabetismo""","""0.15 [0.11, 0.19]""","""0.12 [0.08, 0.15]""","""sí"""
"""pct_sin_religion""","""-0.16 [-0.20, -0.12]""","""-0.10 [-0.14, -0.06]""","""sí"""


**Qué vemos:**

- **12 de las 18 son estables.** Las más fuertes:
  - **densidad** (−0.30 y −0.19): los municipios más densos votan menos de lo que su historia anticipa;
  - **participación económica de las mujeres** (−0.19 y −0.18);
  - **radio** (−0.21 y −0.13);
  - **electores por casilla** (−0.18 y −0.14).
- **Algunos signos contradicen la intuición.** El analfabetismo (+0.15 y +0.12) y la discapacidad van con votar **más** de lo esperado. No es que esas condiciones ayuden a votar: son rasgos de municipios rurales chicos que, a igualdad de historia, votan más en la intermedia. Es el mismo patrón que la pobreza fuera de Oaxaca (notebook 03b, sección 7).
- **No estables:** solteros, automóvil, ocupantes por vivienda, teléfono fijo, desocupación y TV.

## 3 · ¿Mejoran la predicción?

Que una variable se relacione con algo no basta. Debe **agregar información** que el modelo no tiene. Agregamos cada variable (y luego todas) al modelo Huber del notebook 04 y repetimos las dos validaciones.

In [4]:
HISTORIA = ['desviacion_presidencial', 'desviacion_intermedia_previa', 'vecinos_desviacion_presidencial']
LOCALES = ['gobernador_mismo_dia_intermedia', 'diputacion_local_mismo_dia_intermedia', 'ayuntamiento_mismo_dia_intermedia',
           'gobernador_mismo_dia_presidencial', 'diputacion_local_mismo_dia_presidencial', 'ayuntamiento_mismo_dia_presidencial', 'es_sni']
POLITICAS = ['margen_presidencial', 'partidos_efectivos_presidencial']
LISTA = ['pct_rural', 'pct_hombres', 'pct_20_29', 'pct_65_mas', 'crecimiento_lista', 'cobertura_lista_20mas']
SOCIALES = ['escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet', 'pct_residia_otro_pais_hace5', 'pct_residia_otra_entidad_hace5',
            'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud', 'pct_rezago_educativo', 'marginacion', 'pct_localidades_menos_5000',
            'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno']
VIOLENCIA = ['homicidios_100mil_log']
TODAS = HISTORIA + LOCALES + POLITICAS + LISTA + SOCIALES + VIOLENCIA   # las 34 del notebook 04, en el mismo orden de bloques

def huber(): return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), HuberRegressor(alpha=1.0, epsilon=1.35, max_iter=3000))
def boosting(): return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0, random_state=0)
def percentil(x): return (rankdata(x) - 0.5) / len(x) * 100
def error_centrado(y, p): return np.abs((y - np.median(y)) - (p - np.median(p)))
def acierto_20(y, p):
    k = len(y) // 5
    return len(set(np.argsort(y)[:k]) & set(np.argsort(p)[:k])) / k
A15, A21 = con_resultado(2015), con_resultado(2021)
SENTIDOS = [('aprende de 2015, predice 2021', A15, A21), ('aprende de 2021, predice 2015', A21, A15)]
def predecir(modelo, cols, A, B):
    return modelo().fit(A.select(cols).to_numpy().astype(float), A['desviacion_objetivo'].to_numpy()).predict(B.select(cols).to_numpy().astype(float))
base = {n: predecir(huber, TODAS, A, B) for n, A, B in SENTIDOS}

def comparar(etiqueta, modelo, cols, veces=1000):
    fila = {'Modelo': etiqueta}
    for n, A, B in SENTIDOS:
        y = B['desviacion_objetivo'].to_numpy(); p = predecir(modelo, cols, A, B)
        dif = error_centrado(y, p) - error_centrado(y, base[n])
        bs = [dif[i].mean() for i in (rng.integers(0, len(dif), len(dif)) for _ in range(veces))]
        corto = '15→21' if n.startswith('aprende de 2015') else '21→15'
        fila[f'Error {corto} (pp)'] = round(error_centrado(y, p).mean(), 2)
        fila[f'Cambio {corto} [IC 95 %]'] = f'{dif.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]'
        fila[f'Acierto 20 % {corto}'] = round(acierto_20(y, p) * 100, 1)
        fila[f'_mejora_{corto}'] = bool(np.percentile(bs, 97.5) < 0)
    return fila

filas = [comparar('Modelo actual (34 variables)', huber, TODAS)]
filas += [comparar(f'+ {v}', huber, TODAS + [v]) for v in NUEVAS]
filas.append(comparar(f'+ las {estabilidad["estable"].sum()} estables', huber, TODAS + estabilidad.filter(pl.col('estable'))['variable'].to_list()))
filas.append(comparar('+ las 18 nuevas', huber, TODAS + NUEVAS))
filas.append(comparar('Gradient boosting, 34 actuales', boosting, TODAS))
filas.append(comparar('Gradient boosting + las 18 nuevas', boosting, TODAS + NUEVAS))
resultado = pl.DataFrame(filas)
mejoran_en_las_dos = resultado.filter(pl.col('_mejora_15→21') & pl.col('_mejora_21→15'))['Modelo'].to_list()
print('Mejoran el error, fuera del ruido, en las dos validaciones:', mejoran_en_las_dos)
resultado.drop('_mejora_15→21', '_mejora_21→15')

Mejoran el error, fuera del ruido, en las dos validaciones: ['+ promedio_ocupantes_vivienda', '+ densidad_log']


Modelo,Error 15→21 (pp),Cambio 15→21 [IC 95 %],Acierto 20 % 15→21,Error 21→15 (pp),Cambio 21→15 [IC 95 %],Acierto 20 % 21→15
str,f64,str,f64,f64,str,f64
"""Modelo actual (34 variables)""",5.49,"""+0.000 [+0.000, +0.000]""",71.5,6.65,"""+0.000 [+0.000, +0.000]""",57.3
"""+ pct_sin_religion""",5.43,"""-0.061 [-0.072, -0.050]""",72.3,6.69,"""+0.039 [+0.031, +0.048]""",57.1
"""+ pct_catolica""",5.45,"""-0.048 [-0.062, -0.033]""",71.7,6.65,"""+0.003 [+0.001, +0.005]""",57.8
"""+ pct_desocupacion""",5.49,"""+0.001 [-0.002, +0.004]""",71.5,6.69,"""+0.038 [+0.025, +0.053]""",57.3
"""+ tasa_participacion_economica""",5.52,"""+0.030 [+0.003, +0.057]""",71.0,6.67,"""+0.015 [+0.011, +0.020]""",57.6
"""+ tasa_participacion_economica_mujeres""",5.53,"""+0.037 [-0.000, +0.078]""",71.7,6.65,"""+0.004 [+0.001, +0.006]""",57.8
"""+ pct_hogares_jefatura_femenina""",5.54,"""+0.043 [+0.007, +0.077]""",71.7,6.63,"""-0.025 [-0.029, -0.021]""",57.3
"""+ pct_casados_12mas""",5.57,"""+0.073 [+0.053, +0.094]""",71.0,6.64,"""-0.015 [-0.020, -0.011]""",57.8
"""+ pct_solteros_12mas""",5.55,"""+0.055 [+0.034, +0.078]""",70.4,6.65,"""-0.005 [-0.006, -0.003]""",57.8


**Cómo leer la tabla:** "Cambio" es la diferencia de error contra el modelo actual, en puntos. **Negativo = mejora.** El intervalo viene de un bootstrap pareado sobre los municipios de la elección probada: si no cruza el cero, la diferencia no es ruido.

**Qué vemos:**

- **Solo dos variables mejoran en las dos validaciones, y muy poco:**
  - la densidad: el error baja 0.05 pp al predecir 2021 y 0.03 pp al predecir 2015, menos del 1 %;
  - los ocupantes por vivienda: 0.001 y 0.03 pp.

  El acierto en el 20 % de mayor riesgo casi no cambia.
- **Las demás empeoran al menos una validación.** Algunas, como el automóvil o el % sin religión, mejoran 2021 pero empeoran 2015: se acomodan a una elección, no siguen un patrón. Otras, como la participación económica o la desocupación, empeoran o no cambian en las dos.
- **Con las 18 juntas:** al predecir 2021 la diferencia está dentro del ruido; al predecir 2015 mejora 0.12 pp. No pasa la regla.
- **Gradient boosting es mucho peor:** 7.3 puntos de error al predecir 2021 contra 5.5, y su acierto en el 20 % de mayor riesgo cae de 72 % a 42 %. Más variables no lo arreglan. Con dos elecciones para aprender, un modelo flexible se aprende de memoria la elección de entrenamiento.

**Por qué variables estables no mejoran la predicción:** dicen lo mismo que variables que el modelo ya tiene. Por ejemplo, la densidad casi repite población y lo rural, y el radio repite marginación e internet. El modelo ya tenía esa información.

## 4 · ¿Qué pesa de verdad? Importancia por permutación

Para cada variable, y luego para cada bloque, "revolvemos" sus valores entre municipios en la elección que el modelo **no vio** y medimos cuánto sube el error. Si sube mucho, el modelo depende de ella. Se repite 30 veces y se promedia.

Los bloques son más confiables que las variables sueltas: cuando dos variables dicen lo mismo, revolver una sola casi no daña, porque la otra la sustituye.

In [5]:
def importancia(A, B, grupos, veces=30):
    m = huber().fit(A.select(TODAS).to_numpy().astype(float), A['desviacion_objetivo'].to_numpy())
    X, y = B.select(TODAS).to_numpy().astype(float), B['desviacion_objetivo'].to_numpy()
    e0 = error_centrado(y, m.predict(X)).mean()
    salida = {}
    for g, vs in grupos.items():
        idx = [TODAS.index(v) for v in vs]; aumentos = []
        for _ in range(veces):
            Xp = X.copy(); Xp[:, idx] = X[rng.permutation(len(X))][:, idx]
            aumentos.append(error_centrado(y, m.predict(Xp)).mean() - e0)
        salida[g] = np.mean(aumentos)
    return salida
BLOQUES = {'Historia del municipio (3)': HISTORIA, 'Elecciones locales el mismo día y usos y costumbres (7)': LOCALES,
           'Sociales y económicas (15)': SOCIALES, 'Lista nominal (6)': LISTA, 'Competencia política (2)': POLITICAS, 'Violencia (1)': VIOLENCIA}
imp_bloques = {n: importancia(A, B, BLOQUES) for n, A, B in SENTIDOS}
pl.DataFrame({'Bloque': list(BLOQUES),
              'Aumento del error al revolverlo, prediciendo 2021 (pp)': [round(imp_bloques[SENTIDOS[0][0]][b], 2) for b in BLOQUES],
              'Ídem, prediciendo 2015 (pp)': [round(imp_bloques[SENTIDOS[1][0]][b], 2) for b in BLOQUES]})

Bloque,"Aumento del error al revolverlo, prediciendo 2021 (pp)","Ídem, prediciendo 2015 (pp)"
str,f64,f64
"""Historia del municipio (3)""",7.0,5.59
"""Elecciones locales el mismo día y usos y costumbres (7)""",2.87,3.76
"""Sociales y económicas (15)""",1.85,1.49
"""Lista nominal (6)""",0.07,0.19
"""Competencia política (2)""",0.01,0.04
"""Violencia (1)""",-0.0,-0.0


In [6]:
imp_variables = {n: importancia(A, B, {v: [v] for v in TODAS}) for n, A, B in SENTIDOS}
(pl.DataFrame({'Variable': TODAS, 'Prediciendo 2021 (pp)': [imp_variables[SENTIDOS[0][0]][v] for v in TODAS],
               'Prediciendo 2015 (pp)': [imp_variables[SENTIDOS[1][0]][v] for v in TODAS]})
 .with_columns(pl.exclude('Variable').round(2)).sort('Prediciendo 2021 (pp)', descending=True).head(12))

Variable,Prediciendo 2021 (pp),Prediciendo 2015 (pp)
str,f64,f64
"""desviacion_presidencial""",5.35,5.0
"""ayuntamiento_mismo_dia_intermedia""",1.9,7.21
"""gobernador_mismo_dia_presidencial""",0.61,0.0
"""ayuntamiento_mismo_dia_presidencial""",0.47,-0.08
"""desviacion_intermedia_previa""",0.38,0.17
"""gobernador_mismo_dia_intermedia""",0.32,0.07
"""es_sni""",0.19,0.28
"""diputacion_local_mismo_dia_presidencial""",0.17,-0.05
"""poblacion_log""",0.12,1.06


**Qué vemos:**

- **La historia del municipio es lo que más pesa:** sin ella, el error sube entre 5.6 y 7 pp. Sobre todo pesa su posición en la presidencial anterior.
- **Le siguen las elecciones locales el mismo día y usos y costumbres** (unos 3 a 4 pp). Elegir ayuntamiento ese día es, por sí sola, la segunda variable más importante.
- **Las variables sociales y económicas importan como conjunto** (unos 1.5 a 1.9 pp), pero ninguna sola es indispensable, porque se sustituyen entre sí. Por eso agregar más variables de ese tipo no mejora.
- **La competencia política y la violencia casi no pesan** (menos de 0.05 pp), igual que en el notebook 03b.

## 5 · Conclusión y decisión pendiente

**Lo que podemos decir ante el jurado:** probamos 18 variables más de otras fuentes, con hipótesis escritas antes, y un modelo no lineal. Ninguna mejora la predicción de forma relevante. El modelo **no quedó corto de información**: lo que predice la abstención en una intermedia es la historia del municipio y qué se elige ese día. Lo social ya está bien representado.

**D20 (pendiente de decisión).** ¿Agregamos la densidad (y los ocupantes por vivienda) al modelo?

| Opción | A favor | En contra |
|---|---|---|
| (a) Dejar el modelo como está | Ya está validado y es más simple; la mejora es menor al 1 % | — |
| (b) Agregar las dos variables | Mejoran en las dos validaciones, fuera del ruido; la densidad es fácil de explicar | El semáforo casi no cambiaría; hay que volver a correr los notebooks 04 y 05 |

**Recomendación: (a).** La diferencia es real, pero demasiado chica para que valga un cambio a 10 días de la entrega.

**Fuentes que no probamos y por qué:**

- **Participación en la consulta popular de 2021, la revocación de mandato de 2022 y la elección judicial de 2025** (INE). Miden justo la participación sin elección local, así que podrían ser buenas señales para 2027. Pero no hay nada parecido antes de 2015 o 2021, así que **no se pueden validar** con el pasado. Podrían ir al mapa como capa de contexto.
- **Violencia contra autoridades y candidaturas** (bases de organizaciones civiles). Empiezan en 2018, así que solo servirían para una de las dos validaciones.